# UC-DIM-3 — Pentadal Rainfall Dimension Pagination

**Persona:** GIS Specialist — CHIRPS Rainfall Analyst

**Goal:** Page through all 72 pentadal periods of year 2024 using OGC `links[rel=next]`
pagination, verify the February P6 edge case (leap year), and confirm that an oversize
`limit` is clamped rather than rejected.

**Use case:** UC-DIM-3 — CHIRPS pentadal precipitation monitoring

**Prerequisites:**
- `extension_dimensions` installed and registered
- `pentadal-monthly` dimension available (72 periods bounded to 2024)

In [ ]:
import os

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
TOKEN = os.environ.get("DYNASTORE_TOKEN", "")
DIM_ID = "pentadal-monthly"   # 5-day monthly, 72 periods, bounded 2024
DIM_BASE = "/dimensions"

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Accept": "application/json",
}
client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=30.0, follow_redirects=True)
print(f"BASE_URL: {BASE_URL}")
print(f"DIM_ID  : {DIM_ID}")

## Step 1 — Introspect dimension encoding

Confirm `pentadal-monthly` uses a 5-day period provider before attempting to retrieve 72 periods.

In [ ]:
resp = client.get(DIM_BASE)
print(resp.status_code)
assert resp.status_code == 200, f"Expected 200, got {resp.status_code}: {resp.text}"

collections = resp.json().get("collections", [])
dim_meta = next((d for d in collections if d["id"] == DIM_ID), None)
assert dim_meta is not None, f"Dimension '{DIM_ID}' not found"

provider = dim_meta.get("provider", {})
period_days = provider.get("config", {}).get("period_days")
prov_type = provider.get("type", "")

print(f"id          : {dim_meta.get('id')}")
print(f"title       : {dim_meta.get('title')}")
print(f"provider    : {prov_type}")
print(f"period_days : {period_days}")
print(f"invertible  : {provider.get('invertible')}")

assert prov_type == "daily-period", f"Expected daily-period provider, got {prov_type!r}"
assert period_days == 5, f"Expected 5-day periods for pentadal, got {period_days}"
print("\nPentadal temporal dimension confirmed.")

## Step 2 — Page through all 72 periods

Fetch items in pages of 12 following `links[rel=next]` until no next link is present.

In [ ]:
all_periods = []
url = f"{DIM_BASE}/{DIM_ID}/items"
params = {"limit": 12, "offset": 0}
page_num = 0

while True:
    resp = client.get(url, params=params)
    assert resp.status_code == 200, (
        f"Page {page_num}: Expected 200, got {resp.status_code}: {resp.text}"
    )
    data = resp.json()
    page_features = data.get("features", [])
    all_periods.extend(page_features)
    page_num += 1
    print(f"  Page {page_num}: {len(page_features)} periods (running total: {len(all_periods)})")

    next_link = next(
        (lnk["href"] for lnk in data.get("links", []) if lnk.get("rel") == "next"),
        None,
    )
    if not next_link:
        break
    # Follow next link — extract new offset
    from urllib.parse import urlparse, parse_qs
    qs = parse_qs(urlparse(next_link).query)
    params = {"limit": int(qs.get("limit", [12])[0]), "offset": int(qs.get("offset", [0])[0])}

print(f"\nTotal periods collected: {len(all_periods)}")
assert len(all_periods) == 72, (
    f"Expected 72 pentadal periods for the full dataset, got {len(all_periods)}"
)
print("Assertion passed: 72 pentadal periods retrieved via pagination.")

## Step 3 — Verify February edge case

In a pentadal calendar, February has 6 pentads (P7–P12 counting from year start). P12 covers Feb 26–29 in a leap year.

In [ ]:
feb_periods = [
    p for p in all_periods
    if str(p.get("id", "")).startswith("2024-P")
    and 7 <= int(p["id"].split("P")[1]) <= 12
]

print(f"February 2024 pentads ({len(feb_periods)}):")
for p in feb_periods:
    props = p["properties"]
    interval = props.get("time", {}).get("interval", [None, None])
    print(f"  {p['id']}  {interval[0]} — {interval[1]}")

assert len(feb_periods) == 6, (
    f"Expected 6 pentadal periods for February 2024, got {len(feb_periods)}"
)

# Locate P12 (February P6) and verify leap-year end date
p12 = next((p for p in feb_periods if p["id"] == "2024-P12"), None)
if p12:
    end = p12["properties"].get("time", {}).get("interval", [None, None])[1]
    assert end == "2024-02-29", f"Expected 2024-02-29 for leap-year Feb P6, got {end!r}"
    print(f"\nLeap-year edge case confirmed: P12 ends on {end}")

## Edge case — Oversize limit clamped

Passing `limit=99999` must return 200 (server clamps silently), not 422.

In [ ]:
resp = client.get(
    f"{DIM_BASE}/{DIM_ID}/items",
    params={"limit": 10000},
)
print(resp.status_code)
assert resp.status_code == 200, (
    f"Expected 200 for large limit (max=10000), got {resp.status_code}: {resp.text}"
)

data = resp.json()
clamped_features = data.get("features", [])
print(f"Members returned with limit=10000: {len(clamped_features)}")

# All 72 pentadal periods should be returned (dataset smaller than max limit)
assert len(clamped_features) == 72, (
    f"Expected 72 items (full dataset), got {len(clamped_features)}"
)
print("Large limit handled correctly — all 72 periods returned.")